# Preparation

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/crowdflower-weather-twitter/sampleSubmission.csv
/kaggle/input/competitions/crowdflower-weather-twitter/variableNames.txt
/kaggle/input/competitions/crowdflower-weather-twitter/train.csv
/kaggle/input/competitions/crowdflower-weather-twitter/test.csv


In [2]:
# Import libraries
import keras
import re
import shutil
import sklearn
import string
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Import dependencies
from keras import layers
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Set up charts and columns configuration
pd.options.display.max_rows = 2000
pd.options.display.max_columns = 500

2026-05-29 08:46:53.755808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780044414.078216      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780044414.161941      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780044414.868377      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780044414.868428      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780044414.868432      16 computation_placer.cc:177] computation placer alr

# Examine the data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/train.csv')
test = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/test.csv')

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77946 entries, 0 to 77945
Data columns (total 28 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        77946 non-null  int64  
 1   tweet     77946 non-null  object 
 2   state     77946 non-null  object 
 3   location  66994 non-null  object 
 4   s1        77946 non-null  float64
 5   s2        77946 non-null  float64
 6   s3        77946 non-null  float64
 7   s4        77946 non-null  float64
 8   s5        77946 non-null  float64
 9   w1        77946 non-null  float64
 10  w2        77946 non-null  float64
 11  w3        77946 non-null  float64
 12  w4        77946 non-null  float64
 13  k1        77946 non-null  float64
 14  k2        77946 non-null  float64
 15  k3        77946 non-null  float64
 16  k4        77946 non-null  float64
 17  k5        77946 non-null  float64
 18  k6        77946 non-null  float64
 19  k7        77946 non-null  float64
 20  k8        77946 non-null  fl

In [5]:
# Display the number of indexes in the train dataset
for i in range(2):
    n = np.random.randint(77000)
    sample = train['tweet'][n]
    print('-' * 80)
    print(f'Index: {n}; \nSample: \n{sample}')

--------------------------------------------------------------------------------
Index: 19026; 
Sample: 
@mention I hope u do feel better soon..have u had crazy weather changes?that always kils me! Pretty good day ready for weekend!
--------------------------------------------------------------------------------
Index: 33219; 
Sample: 
Met up with an awesome advertising duo. They really made my day! That and #pdx mixed bag weather haha.


In [6]:
test.info()
test.sample(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42157 entries, 0 to 42156
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        42157 non-null  int64 
 1   tweet     42157 non-null  object
 2   state     14323 non-null  object
 3   location  42144 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.3+ MB


,id,tweet,state,location
14716,41670,@ArchBishopofHH I have been waiting for this w...,NaN,Virginia Beach
8076,22971,Severe Thunderstorm Warning for Clinton County...,NaN,"Lansing, MI"


In [7]:
# Display the number of indexes in the test dataset
for i in range(2):
    n = np.random.randint(40000)
    sample = test['tweet'][n]
    print('-' * 80)
    print(f'Index: {n}; \nSample: \n{sample}')

--------------------------------------------------------------------------------
Index: 16555; 
Sample: 
RT @imGeezyhoe: Nice and hot out, whole block out
--------------------------------------------------------------------------------
Index: 17065; 
Sample: 
Weather


# Sentence standardization

In [8]:
# Define configurations
def custom_standardization(sentence):
    sample = tf.strings.lower(sentence)
    stripped_html = tf.strings.regex_replace(sample, 'link', '')
    return tf.strings.regex_replace(stripped_html, '[%s]'%re.escape(string.punctuation), '')

max_features = 10000
sequence_length = 250

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)
vectorize_layer.adapt(train['tweet'])

for i in range(1):
    sample = np.random.randint(77000)
    print(f"Before standardization: \n{train['tweet'][sample]}")
    print('-' * 80)
    print(f"After standardization: \n{custom_standardization(train['tweet'][sample])}")
    print('-' * 80)
    print(f"After vectorization: \n{vectorize_layer(train['tweet'][sample])}")
    print('*' * 80)

2026-05-29 08:47:14.513982: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Before standardization: 
i want a pet snow leopard
--------------------------------------------------------------------------------
After standardization: 
b'i want a pet snow leopard'
--------------------------------------------------------------------------------
After vectorization: 
[   9  145    7 3797   55 6474    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
  

# Model training

In [9]:
x = train.iloc[:, 1:2].values
X = np.array(vectorize_layer(x))
X.shape

(77946, 250)

In [10]:
Y = np.array(train.iloc[:, 4:])
Y.shape

(77946, 24)

In [11]:
a = test.iloc[:, 1:2].values
A = np.array(vectorize_layer(a))
A.shape

(42157, 250)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3)
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print('-' * 50)
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

X_train: (54562, 250), y_train: (54562, 24)
--------------------------------------------------
X_test: (23384, 250), y_test: (23384, 24)


In [13]:
# Train the model using Lasso function
from sklearn.linear_model import Lasso

model = Lasso()

_ = model.fit(X_train, y_train)
y_pred = model.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(0.24794578275755189)

In [14]:
# Predict the model
predict = model.predict(A)
predict

array([[0.06214019, 0.26956886, 0.28130401, ..., 0.1602593 , 0.01057931,
        0.06927684],
       [0.02742936, 0.11831682, 0.53399487, ..., 0.09551432, 0.02618099,
        0.12089733],
       [0.04443055, 0.15929161, 0.46847658, ..., 0.11094828, 0.01747752,
        0.13090454],
       ...,
       [0.04655954, 0.23883647, 0.31004109, ..., 0.15615437, 0.0273232 ,
        0.05053804],
       [0.05434731, 0.24145271, 0.31105121, ..., 0.1402067 , 0.01494243,
        0.06000327],
       [0.06230551, 0.27727017, 0.26138764, ..., 0.1670544 , 0.01043352,
        0.05361627]], shape=(42157, 24))

# File submission

In [15]:
# Create a submission file
submission = pd.read_csv('/kaggle/input/competitions/crowdflower-weather-twitter/sampleSubmission.csv')
ids = test['id']
label = pd.DataFrame(predict, columns=submission.iloc[:, 1:].columns)
submission = pd.concat([ids, label], axis=1)

In [16]:
submission

,id,s1,s2,s3,s4,s5,w1,w2,w3,w4,k1,k2,k3,k4,k5,k6,k7,k8,k9,k10,k11,k12,k13,k14,k15
0,4,0.062140,0.269569,0.281304,0.255360,0.129380,0.770992,0.079242,0.120929,0.032512,0.021244,0.116354,0.008256,0.135803,0.052415,0.001458,0.288795,0.003539,0.076802,0.100788,0.029199,0.111881,0.160259,0.010579,0.069277
1,5,0.027429,0.118317,0.533995,0.091626,0.231750,0.668268,0.090104,0.146892,0.087025,0.022827,0.053626,0.008595,0.065403,0.086472,0.001458,0.263753,0.003539,0.095727,0.096470,0.054041,0.242741,0.095514,0.026181,0.120897
2,7,0.044431,0.159292,0.468477,0.134355,0.194748,0.715637,0.077597,0.150369,0.058436,0.022477,0.084567,0.009784,0.096097,0.116904,0.001458,0.288093,0.003539,0.096537,0.079614,0.035939,0.158973,0.110948,0.017478,0.130905
3,8,0.049749,0.207534,0.389767,0.190855,0.161582,0.747409,0.080118,0.129599,0.046208,0.021621,0.098506,0.008177,0.108657,0.090401,0.001458,0.286448,0.003539,0.084771,0.093118,0.034328,0.140874,0.134669,0.016137,0.097968
4,12,0.051448,0.226026,0.333042,0.222802,0.166363,0.724814,0.088380,0.130561,0.057312,0.025524,0.095003,0.007470,0.121300,0.041558,0.001458,0.284686,0.003539,0.077179,0.098699,0.036082,0.163876,0.147847,0.018091,0.058381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42152,120094,0.030581,0.110018,0.567162,0.102142,0.192133,0.694493,0.087336,0.136980,0.078264,0.021903,0.051730,0.008782,0.076553,0.134234,0.001458,0.256133,0.003539,0.104998,0.083955,0.048776,0.237438,0.103367,0.023484,0.142395
42153,120096,0.064053,0.293055,0.238034,0.284047,0.117774,0.779271,0.081585,0.113848,0.028871,0.022666,0.119311,0.007202,0.143276,0.026431,0.001458,0.285577,0.003539,0.069157,0.106464,0.029502,0.116660,0.171469,0.010753,0.043042
42154,120099,0.046560,0.238836,0.310041,0.224218,0.177707,0.705194,0.109732,0.118357,0.063979,0.024350,0.087861,0.007878,0.124122,0.026379,0.001458,0.239822,0.003539,0.078297,0.109068,0.034309,0.182256,0.156154,0.027323,0.050538
42155,120101,0.054347,0.241453,0.311051,0.213941,0.178710,0.728248,0.086105,0.136347,0.048470,0.022133,0.099671,0.007482,0.113022,0.034063,0.001458,0.286754,0.003539,0.074558,0.106369,0.040189,0.143937,0.140207,0.014942,0.060003


In [17]:
print("Successfully saved as CSV file.")

Successfully saved as CSV file.
